# MMT Demand Forecasting Pipeline

Trains per-partition XGBRegressor models using **Feature Store** features.
Training runs as a **Snowflake ML Job** on a compute pool.
Models logged to **Model Registry**, forecasts to `SCORING.DEMAND_FORECASTS`.


In [ ]:
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F
from snowflake.snowpark.window import Window
from snowflake.ml.feature_store import FeatureStore, CreationMode

session = Session.builder.configs({"connection_name": "my_connection"}).create()
session.use_database('SB_COMMAND_CENTER')
session.use_warehouse('COMPUTE_WH')

fs = FeatureStore(
    session=session,
    database='SB_COMMAND_CENTER',
    name='FEATURE_STORE',
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST
)
print("Session + Feature Store ready")

## Read POS Features from Feature Store

Pull weekly POS metrics from `POS_WEEKLY_SALES_FV` — the governed, versioned
feature view created in notebook 01. Then add lag/rolling/target columns on top.


In [ ]:
# Get the registered feature view
pos_fv = fs.get_feature_view('POS_WEEKLY_SALES_FV', 'v1')
print(f"Feature view: {pos_fv.name}${pos_fv.version}")
print(f"Features: {pos_fv.feature_names}")

# Read feature data as a Snowpark DataFrame
pos_features = fs.read_feature_view(pos_fv)
pos_features = pos_features.with_column(
    'RETAILER_BRAND_KEY',
    F.concat(F.col('RETAILER_NAME'), F.lit('_'), F.col('ITEM_BRAND'))
)
print(f"POS feature rows: {pos_features.count()}")

## Add Lag, Rolling, and Target Columns

The Feature Store provides the base metrics (units, revenue, inventory, markdown, etc.).
We add time-series features (lags, rolling averages, seasonality) and the prediction target
on top of the governed features.


In [ ]:
w = Window.partition_by('RETAILER_BRAND_KEY').order_by('RETAILER_DATE')

training_df = (pos_features
    .with_column('LAG_1W', F.lag('GROSS_UNITS_SOLD', 1).over(w))
    .with_column('LAG_2W', F.lag('GROSS_UNITS_SOLD', 2).over(w))
    .with_column('LAG_3W', F.lag('GROSS_UNITS_SOLD', 3).over(w))
    .with_column('LAG_4W', F.lag('GROSS_UNITS_SOLD', 4).over(w))
    .with_column('LAG_8W', F.lag('GROSS_UNITS_SOLD', 8).over(w))
    .with_column('ROLLING_4W_AVG',
        F.avg('GROSS_UNITS_SOLD').over(
            Window.partition_by('RETAILER_BRAND_KEY').order_by('RETAILER_DATE').rows_between(-4, -1)))
    .with_column('ROLLING_8W_AVG',
        F.avg('GROSS_UNITS_SOLD').over(
            Window.partition_by('RETAILER_BRAND_KEY').order_by('RETAILER_DATE').rows_between(-8, -1)))
    .with_column('ROLLING_13W_AVG',
        F.avg('GROSS_UNITS_SOLD').over(
            Window.partition_by('RETAILER_BRAND_KEY').order_by('RETAILER_DATE').rows_between(-13, -1)))
    .with_column('WEEK_OF_YEAR', F.weekofyear('RETAILER_DATE'))
    .with_column('MONTH_NUM', F.month('RETAILER_DATE'))
    .with_column('IS_HOLIDAY_SEASON',
        F.iff(F.col('MONTH_NUM').isin([11, 12, 1]), F.lit(1), F.lit(0)))
    .with_column('IS_SUMMER',
        F.iff(F.col('MONTH_NUM').isin([6, 7, 8]), F.lit(1), F.lit(0)))
    .with_column('TARGET_UNITS_NEXT_WEEK', F.lead('GROSS_UNITS_SOLD', 1).over(w))
    .filter(
        (F.col('LAG_8W').is_not_null()) & (F.col('TARGET_UNITS_NEXT_WEEK').is_not_null())
    )
)

# Materialize for the ML Job to read
session.sql('CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.FORECASTING').collect()
training_df.write.mode('overwrite').save_as_table('SB_COMMAND_CENTER.FORECASTING.TRAINING_DATA')
row_count = session.table('SB_COMMAND_CENTER.FORECASTING.TRAINING_DATA').count()
print(f"Training data materialized: {row_count} rows")

In [ ]:
# Verify partition sizes
session.table('SB_COMMAND_CENTER.FORECASTING.TRAINING_DATA').group_by(
    'RETAILER_BRAND_KEY'
).agg(
    F.count('*').alias('WEEKS'),
    F.min('RETAILER_DATE').alias('START_DT'),
    F.max('RETAILER_DATE').alias('END_DT')
).sort('RETAILER_BRAND_KEY').show(10)

## Submit Training as ML Job

Uses `@remote` to run XGBoost training on a Snowflake compute pool.
Trains one model per retailer×brand partition, pickles to stage, catalogs metrics.


In [ ]:
session.sql('CREATE STAGE IF NOT EXISTS SB_COMMAND_CENTER.FORECASTING.ML_STAGE').collect()
session.sql('CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.SCORING').collect()
session.sql('CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.REGISTRY').collect()

from snowflake.ml.jobs import remote

@remote(
    compute_pool='ML_CPU_POOL',
    stage_name='SB_COMMAND_CENTER.FORECASTING.ML_STAGE',
    pip_requirements=['xgboost', 'scikit-learn', 'pandas', 'numpy'],
    session=session,
)
def train_and_score():
    import pickle, json, tempfile, os
    import numpy as np
    import pandas as pd
    from datetime import datetime
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
    from snowflake.snowpark import Session as SpSession
    from snowflake.ml.registry import Registry

    sp = SpSession.builder.getOrCreate()

    FEATURE_COLS = [
        'LAG_1W', 'LAG_2W', 'LAG_3W', 'LAG_4W', 'LAG_8W',
        'ROLLING_4W_AVG', 'ROLLING_8W_AVG', 'ROLLING_13W_AVG',
        'WEEK_OF_YEAR', 'MONTH_NUM', 'IS_HOLIDAY_SEASON', 'IS_SUMMER',
        'MARKDOWN_PCT', 'INVENTORY_ON_HAND', 'ON_ORDER_QUANTITY'
    ]
    TARGET_COL = 'TARGET_UNITS_NEXT_WEEK'

    df = sp.table('SB_COMMAND_CENTER.FORECASTING.TRAINING_DATA').to_pandas()
    partitions = df['RETAILER_BRAND_KEY'].unique()
    version_id = f"v{datetime.now().strftime('%Y%m%d_%H%M')}"

    sp.sql("""CREATE TABLE IF NOT EXISTS SB_COMMAND_CENTER.FORECASTING.MODEL_CATALOG (
        PARTITION_ID VARCHAR, TRAINED_AT DATE, METRICS VARIANT,
        MODEL_VERSION VARCHAR, IS_ACTIVE BOOLEAN, STAGE_PATH VARCHAR
    )""").collect()

    # ── Phase 1: Train all partition models ──
    all_metrics = []
    models = {}
    for pid in partitions:
        pdf = df[df['RETAILER_BRAND_KEY'] == pid].sort_values('RETAILER_DATE')
        train, val = pdf.iloc[:-8], pdf.iloc[-8:]
        X_train, y_train = train[FEATURE_COLS].fillna(0), train[TARGET_COL]
        X_val, y_val = val[FEATURE_COLS].fillna(0), val[TARGET_COL]

        model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_val)
        metrics = {
            'VAL_MAE': float(mean_absolute_error(y_val, preds)),
            'VAL_RMSE': float(np.sqrt(mean_squared_error(y_val, preds))),
            'VAL_MAPE': float(mean_absolute_percentage_error(y_val, preds)),
        }
        all_metrics.append({'partition': pid, **metrics})
        models[pid] = model

        # Serialize to stage
        stage_path = f"@SB_COMMAND_CENTER.FORECASTING.ML_STAGE/training_{version_id}/{pid}"
        with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
            pickle.dump(model, f)
            tmp = f.name
        sp.file.put(tmp, stage_path, auto_compress=False, overwrite=True)
        os.unlink(tmp)

        sp.sql(f"""INSERT INTO SB_COMMAND_CENTER.FORECASTING.MODEL_CATALOG
            SELECT '{pid}', CURRENT_DATE(), PARSE_JSON('{json.dumps(metrics)}'),
                   '{version_id}', TRUE, '{stage_path}'
        """).collect()

    avg_mape = float(np.mean([m['VAL_MAPE'] for m in all_metrics]))
    avg_rmse = float(np.mean([m['VAL_RMSE'] for m in all_metrics]))

    # ── Phase 2: Log best model to Registry ──
    reg = Registry(sp, database_name='SB_COMMAND_CENTER', schema_name='REGISTRY')
    best = min(all_metrics, key=lambda x: x['VAL_MAPE'])
    best_model = models[best['partition']]
    sample = pd.DataFrame([{c: 0.0 for c in FEATURE_COLS}])

    reg.log_model(
        best_model,
        model_name='demand_forecast_xgb',
        version_name=version_id,
        metrics={'avg_mape': avg_mape, 'avg_rmse': avg_rmse,
                 'n_partitions': len(partitions),
                 'best_partition': best['partition'],
                 'best_mape': best['VAL_MAPE']},
        sample_input_data=sample,
    )

    # ── Phase 3: Generate 8-week forecasts ──
    forecast_records = []
    for pid in partitions:
        pdf = df[df['RETAILER_BRAND_KEY'] == pid].sort_values('RETAILER_DATE')
        model = models[pid]
        last_row = pdf.iloc[-1]
        last_date = pd.Timestamp(last_row['RETAILER_DATE'])
        history = pdf['GROSS_UNITS_SOLD'].tolist()

        for wk in range(1, 9):
            fd = last_date + pd.Timedelta(weeks=wk)
            features = {
                'LAG_1W': history[-1], 'LAG_2W': history[-2],
                'LAG_3W': history[-3], 'LAG_4W': history[-4],
                'LAG_8W': history[-8] if len(history) >= 8 else history[-1],
                'ROLLING_4W_AVG': np.mean(history[-4:]),
                'ROLLING_8W_AVG': np.mean(history[-8:]),
                'ROLLING_13W_AVG': np.mean(history[-13:]),
                'WEEK_OF_YEAR': fd.isocalendar()[1], 'MONTH_NUM': fd.month,
                'IS_HOLIDAY_SEASON': 1 if fd.month in (11,12,1) else 0,
                'IS_SUMMER': 1 if fd.month in (6,7,8) else 0,
                'MARKDOWN_PCT': float(last_row['MARKDOWN_PCT']),
                'INVENTORY_ON_HAND': float(last_row['INVENTORY_ON_HAND']),
                'ON_ORDER_QUANTITY': float(last_row['ON_ORDER_QUANTITY']),
            }
            pred = max(0, float(model.predict(pd.DataFrame([features])[FEATURE_COLS])[0]))
            history.append(pred)
            forecast_records.append({
                'RETAILER_NAME': last_row['RETAILER_NAME'],
                'ITEM_BRAND': last_row['ITEM_BRAND'],
                'RETAILER_BRAND_KEY': pid,
                'FORECAST_DATE': fd.date(),
                'WEEKS_AHEAD': wk,
                'FORECASTED_UNITS': round(pred, 1),
                'MODEL_VERSION': version_id,
            })

    forecast_sdf = sp.create_dataframe(pd.DataFrame(forecast_records))
    forecast_sdf.write.mode('overwrite').save_as_table('SB_COMMAND_CENTER.SCORING.DEMAND_FORECASTS')

    return {
        'version_id': version_id,
        'n_partitions': len(partitions),
        'avg_mape': avg_mape,
        'avg_rmse': avg_rmse,
        'best_partition': best['partition'],
        'best_mape': best['VAL_MAPE'],
        'forecasts_written': len(forecast_records),
    }

print("Training function defined — trains, logs to registry, and generates forecasts on ML_CPU_POOL")

In [ ]:
# Submit and wait — all work happens on the compute pool
job = train_and_score()
result = job.result()

print(f"Training: {result['n_partitions']} partition models")
print(f"Avg MAPE: {result['avg_mape']:.3f}, Avg RMSE: {result['avg_rmse']:.1f}")
print(f"Best: {result['best_partition']} (MAPE={result['best_mape']:.3f})")
print(f"Registry: demand_forecast_xgb/v1 logged")
print(f"Forecasts: {result['forecasts_written']} rows written to SCORING.DEMAND_FORECASTS")

## Verify


In [ ]:
session.table('SB_COMMAND_CENTER.SCORING.DEMAND_FORECASTS').group_by(
    'RETAILER_NAME', 'ITEM_BRAND'
).agg(
    F.count('*').alias('FORECAST_WEEKS'),
    F.round(F.avg('FORECASTED_UNITS'), 0).alias('AVG_FORECAST')
).sort('RETAILER_NAME', 'ITEM_BRAND').show()

In [ ]:
session.sql("""
SELECT PARTITION_ID, TRAINED_AT, MODEL_VERSION,
       METRICS:VAL_MAPE::FLOAT AS mape,
       METRICS:VAL_RMSE::FLOAT AS rmse
FROM SB_COMMAND_CENTER.FORECASTING.MODEL_CATALOG
WHERE IS_ACTIVE = TRUE
ORDER BY mape LIMIT 10
""").show()

In [ ]:
session.sql("SHOW MODELS IN SCHEMA SB_COMMAND_CENTER.REGISTRY").show()